# NASA C-MAPSS Turbofan RUL - Exploratory Data Analysis

**Author**: Sandeep Grover, Independent Research

**Goal**: Inspect FD001 (and the other three subsets) before modelling. Build an empirical understanding of:
1. Schema, dtype, missingness.
2. Distribution of run-lengths (cycles to failure) per engine.
3. Sensor signal behaviour over engine life (drift, monotonicity, noise level).
4. Constant or near-constant sensors that can be dropped.
5. Operational settings vs sensor readings (especially for FD002 / FD004 which have 6 op conditions).
6. Construction of piece-wise linear RUL labels with cap at 125 cycles.

## 1. Setup and data loading

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='talk')

DATA_DIR = Path('../data')
# After unzipping nasa-cmaps.zip the subfolder is usually 'CMaps'
CMAPS_DIR = DATA_DIR / 'CMaps' if (DATA_DIR / 'CMaps').exists() else DATA_DIR
print('Data dir:', CMAPS_DIR.resolve())
print('Files :', sorted(p.name for p in CMAPS_DIR.glob('*.txt')))

In [ ]:
COLS = (
    ['unit_number', 'time_in_cycles']
    + [f'op_setting_{i}' for i in range(1, 4)]
    + [f'sensor_{i}'    for i in range(1, 22)]
)

def load_subset(subset='FD001'):
    train = pd.read_csv(CMAPS_DIR / f'train_{subset}.txt', sep=r'\s+', header=None, names=COLS)
    test  = pd.read_csv(CMAPS_DIR / f'test_{subset}.txt',  sep=r'\s+', header=None, names=COLS)
    rul   = pd.read_csv(CMAPS_DIR / f'RUL_{subset}.txt',   sep=r'\s+', header=None, names=['rul'])
    return train, test, rul

train, test, rul_truth = load_subset('FD001')
print('Train shape :', train.shape)
print('Test  shape :', test.shape)
print('RUL   shape :', rul_truth.shape)

## 2. Schema, dtype, missingness

In [ ]:
train.head()

In [ ]:
train.describe().T

In [ ]:
print(train.dtypes)
print('Missing values per column (train):')
print(train.isna().sum().sum(), 'total NaN cells')

## 3. Run-length distribution

Each engine starts at cycle 1 and degrades until failure. The maximum cycle per engine in the training set IS the true RUL=0 point. We compute per-unit max cycle to characterise the failure-time distribution.

In [ ]:
max_cycles = train.groupby('unit_number')['time_in_cycles'].max()
print(max_cycles.describe())

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(max_cycles, bins=25, color='#38bdf8', edgecolor='#0f172a')
ax.set_xlabel('Cycles to failure')
ax.set_ylabel('Engine count')
ax.set_title('FD001 - Distribution of run-to-failure cycles')
plt.tight_layout()

## 4. Sensor behaviour over engine life

We trace each sensor as a function of cycle for a handful of engines to inspect monotonicity, regime shifts, and noise.

In [ ]:
sensor_cols = [c for c in train.columns if c.startswith('sensor_')]
sample_units = train['unit_number'].drop_duplicates().sample(5, random_state=0).tolist()

fig, axes = plt.subplots(7, 3, figsize=(15, 18), sharex=True)
for ax, sc in zip(axes.flat, sensor_cols):
    for u in sample_units:
        sub = train[train['unit_number'] == u]
        ax.plot(sub['time_in_cycles'], sub[sc], alpha=0.6, lw=0.8)
    ax.set_title(sc, fontsize=10)
    ax.tick_params(labelsize=8)
fig.suptitle('Sensor trajectories - 5 random engines', y=1.0)
plt.tight_layout()

## 5. Identify constant or near-constant sensors

On FD001 several sensors are flat throughout life (no information). We drop them in the modelling stage.

In [ ]:
stds = train[sensor_cols].std().sort_values()
print(stds)
low_var = stds[stds < 1e-3].index.tolist()
print('Drop candidates (std < 1e-3):', low_var)

## 6. Constructing piece-wise linear RUL with cap at 125

Heimes (2008) and Zheng (2017) showed that flat-cap RUL labels (constant value while the engine is healthy, linear decay only near end of life) are easier to learn than a pure linear schedule. The conventional cap is 125 cycles.

In [ ]:
RUL_CAP = 125

def add_piecewise_rul(df, cap=RUL_CAP):
    max_per_unit = df.groupby('unit_number')['time_in_cycles'].transform('max')
    rul_linear   = max_per_unit - df['time_in_cycles']
    df = df.copy()
    df['rul'] = np.minimum(rul_linear, cap)
    return df

train_rul = add_piecewise_rul(train)
print(train_rul[['unit_number','time_in_cycles','rul']].head(15))

fig, ax = plt.subplots(figsize=(9, 4))
for u in sample_units:
    sub = train_rul[train_rul['unit_number'] == u]
    ax.plot(sub['time_in_cycles'], sub['rul'], alpha=0.7, label=f'unit {u}')
ax.set_xlabel('Cycle'); ax.set_ylabel('RUL')
ax.set_title(f'Piece-wise linear RUL (cap = {RUL_CAP})')
ax.legend(fontsize=8)
plt.tight_layout()

## 7. Operational settings

FD001 has a single operating regime, so op_settings should be roughly constant. FD002 and FD004 have 6 distinct regimes; clustering on op_settings is the standard preprocessing step.

In [ ]:
op_cols = [c for c in train.columns if c.startswith('op_setting_')]
print(train[op_cols].describe().T)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, oc in zip(axes, op_cols):
    ax.hist(train[oc], bins=40, color='#a78bfa', edgecolor='#0f172a')
    ax.set_title(oc)
plt.tight_layout()

## 8. EDA take-aways for modelling

1. **Drop low-variance sensors** identified in section 5 - they cannot inform RUL.
2. **Use piece-wise linear RUL with cap = 125** for all training labels (section 6).
3. **Window length 30** matches the literature standard (Zheng 2017, Listou Ellefsen 2019). Last 30 cycles per engine become a single training sample.
4. **For FD002 / FD004**: cluster op_settings into 6 regimes and z-score sensors per regime before sequence modelling.
5. **Train / val split**: hold out a fraction of engines (not random rows) so leakage between cycles of the same engine is impossible.
6. **Evaluation**: compute RMSE, MAE, and the NASA scoring function on the held-out test set against `RUL_FDxxx.txt`.

These design choices feed `src/model_baseline.py` (Random Forest on lag/window stats) and `src/model_advanced.py` (LSTM / 1D-Transformer on raw sequences).